In [ ]:
%%capture

# Node LTS and Mermaid CLI (Skip when need to only train)
!curl -fsSL https://deb.nodesource.com/setup_24.x | sudo -E bash -
!apt-get install -y nodejs -qq
!apt-get update -qq && apt-get install -y chromium-browser libnss3 libgconf-2-4 libfontconfig1 -qq
!npm install -g @mermaid-js/mermaid-cli --silent

In [ ]:
# ===========================
# 5. Custom Mermaid Validation
# ===========================
def validate_mermaid(mmd_code, filename="test.mmd"):
    with open(filename, "w") as f:
        f.write(mmd_code)
    try:
        config_path = "puppeteer-config.json"
        with open(config_path, "w") as f:
            json.dump({"args": ["--no-sandbox", "--disable-setuid-sandbox", "--disable-dev-shm-usage"]}, f)
        result = subprocess.run(
            ["mmdc", "-i", filename, "-o", "test.svg", "-p", config_path],
            capture_output=True,
            text=True,
            timeout=30
        )
        return result.returncode == 0, result.stdout + result.stderr
    except Exception as e:
        return False, str(e)

# Best fix for Gemma-3: Patch the attention mask logic via Unsloth helper
# (Required for Gemma-3 to avoid "attention_mask is None" / NoneType errors in Gemma3Attention_forward)
FastLanguageModel.for_inference(model)

success_count = 0
test_samples = eval_dataset.select(range(min(20, len(eval_dataset))))

for idx, example in enumerate(test_samples):
    try:
        messages = example["messages"]
        user_content = messages[0]["content"]

        prompt = tokenizer.apply_chat_template(
            [{"role": "user", "content": user_content}],
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        with torch.no_grad():
            # Setting use_cache=False is a fallback if broadcasting errors persist
            # but we try True first with patched model
            outputs = model.generate(
                input_ids=inputs.input_ids,
                attention_mask=inputs.attention_mask,
                max_new_tokens=512,
                use_cache=True,
                temperature=0.7,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )

        input_length = inputs.input_ids.shape[1]
        generated = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

        mmd = None
        if "```mermaid" in generated:
            start = generated.find("```mermaid") + 11
            end = generated.find("```", start)
            mmd = generated[start:end].strip()

        if mmd:
            valid, log = validate_mermaid(mmd)
            success_count += int(valid)
            print(f"Sample {idx+1}/20 | Valid: {valid}")
            if not valid:
                 print(f"   Error log excerpt: {log[:100]}...")
        else:
            print(f"Sample {idx+1}/20 | No Mermaid block found")
    except Exception as e:
        print(f"Sample {idx+1}/20 | Error: {e}")

print(f"\n✅ Mermaid compilation success rate: {success_count}/20 = {success_count/20*100:.1f}%")